In [22]:
from dotenv import load_dotenv
_ = load_dotenv()

In [23]:
import json
from pathlib import Path
from runner import PlanAndExecuteRunner
from base import Env, Task
from plan_and_execute_agent import PlanAndExecuteAgent
from langchain_ollama import ChatOllama
from agentic_ai.data.taubench_data import load_taubench_data, TaubenchConnection
from agentic_ai.prompts import TAUBENCH_WIKI
from agentic_ai.tools import ALL_TAUBENCH_TOOLS

# Initialize model
model = ChatOllama(
    model="gpt-oss:20b",
    keep_alive=-1,
    num_ctx=16*1024
)

# Load tasks
file = Path() / "tasks.json"
data = json.loads(file.read_text())

tasks: list[Task] = []
for t in data:
    tasks.append(Task.model_validate(t))
    
assert len(tasks) == 115
print(f"Loaded {len(tasks)} tasks.")

Loaded 115 tasks.


In [24]:
# Instantiate the runner and the agent with interrupts enabled
runner = PlanAndExecuteRunner()
plan_execute_agent = PlanAndExecuteAgent(
    model=model,
    wiki=TAUBENCH_WIKI,
    tools=ALL_TAUBENCH_TOOLS,
    enable_interrupt=False
)

# Run the first task in the environment
with TaubenchConnection.session() as session:
    env = Env(
        task=tasks[0],
        user_strategy="llm",
        tools=ALL_TAUBENCH_TOOLS,
        data_load_func=load_taubench_data,
        data=session.data
    )
    result = runner.run(
        agent=plan_execute_agent,
        env=env,
        verbose=True,
        max_num_steps=50
    )
    print("\nResult:", result)

INSTRUCTION: You are Yusuf Rossi in 19122. You received your order #W2378156 and wish to exchange the mechanical keyboard for a similar one but with clicky switches and the smart thermostat for one compatible with Google Home instead of Apple HomeKit. If there is no keyboard that is clicky, RGB backlight, full size, you'd go for no backlight. You are detail-oriented and want to make sure everything is addressed in one go.
================================== Ai Message ==================================

Hi, how can I help you today?
================================ Human Message =================================

I received order #W2378156 and would like to exchange the mechanical keyboard for a full‑size clicky‑switch model (if no RGB backlight is available, just one without backlight), and swap the smart thermostat for a version compatible with Google Home instead of Apple HomeKit.
================================== Ai Message ==================================
Name: planner
Tool Call